# 🏗️ System Design — Ultra-Elaborate Mental Models

> **Every section answers four questions: WHY this exists, WHAT it is, HOW it works, WHEN to use it.**
> Each topic includes: real-world scenarios from production companies, ❌ before / ✅ after code, and *"Where this is seen in frameworks"* callouts.

---
**Topics**
1. The Architect's Decision Framework
2. Thinking About Scale — Numbers Every Engineer Knows
3. CAP Theorem & Consistency Spectrum
4. Load Balancing & Consistent Hashing
5. Caching Deep Dive — All Layers
6. Data Partitioning & Replication
7. Reliability Patterns — The Production Survival Kit
8. API Design at Scale (REST · gRPC · GraphQL)
9. Monolith → Microservices Spectrum
10. Clean / Hexagonal Architecture — Structure That Survives
11. Event-Driven, CQRS & Event Sourcing
12. Real-World Case Studies (Netflix · Stripe · Twitter/X · Uber)
13. The Interview Playbook — Back-of-Envelope + Framework

---
## 1 · The Architect's Decision Framework

### 🧠 Mental Model — *The Trade-off Lens*

> **Every architectural decision trades one property for another. There is no free lunch. The senior engineer names the trade-off; the junior engineer ignores it.**

| Question | What it reveals |
|---|---|
| What are the functional requirements? | What the system *does* |
| What are the non-functional requirements? | How well it does it (scale, latency, consistency) |
| What is the read/write ratio? | Whether to optimize reads (cache) or writes (queue/batch) |
| What fails, and how badly? | Where to invest in resilience |
| What's the consistency requirement? | CP vs AP, sync vs async |

### A Repeatable 7-Step Framework for Any Design Question

```
1. CLARIFY   → Functional requirements, then scale (DAU, QPS, p99 latency)
2. ESTIMATE  → QPS, storage/year, bandwidth — find the BOTTLENECK
3. API       → Define the contract (it constrains everything downstream)
4. DATA      → Schema, SQL vs NoSQL, access patterns
5. HIGH LEVEL→ LB → stateless app → cache → DB + async workers
6. DEEP DIVE → Scale the bottleneck (cache? shard? replicate?)
7. OPERATE   → Failures, timeouts, retries, circuit breakers, monitoring
```

### 🌍 Real-World: How Companies Apply This
- **Stripe**: Step 1 reveals payments need CP consistency; step 6 reveals idempotency keys, not sharding, as the bottleneck
- **Twitter**: Step 1 reveals fan-out is the bottleneck (celebrity with 50M followers tweets once); step 6 leads to pre-computed timelines (write-fan-out) vs pull (read-fan-out) trade-off
- **Netflix**: Step 1 reveals read-heavy streaming; step 6 leads to CDN edge caching, not database scaling

### ⚠️ The Two Cardinal Sins
1. **Over-engineering** before you know the bottleneck (premature sharding)
2. **Under-engineering** by ignoring failure modes (no retry + idempotency = double-charges)

---
## 2 · Thinking About Scale — Numbers Every Engineer Knows

### 🧠 Mental Model — *The Ruler*

> **Before you can reason about a design, you need a ruler. Latency numbers are that ruler. If you don't know the cost of a cross-DC call vs a memory read, your design is guesswork.**

**WHY**: You need these numbers to spot when a design is physically impossible (e.g., calling a cross-region API 100 times per user request and still hitting a 10ms p99).

**WHAT**: The relative cost of every common I/O operation, ordered by magnitude.

**HOW**: Use these to catch anti-patterns — N+1 queries, synchronous chains of slow calls, missing caches.

**WHEN**: Every time you design a new request path. Ask: what's the slowest thing in this chain?

### The Latency Ruler

| Operation | Latency | Human analogy |
|---|---|---|
| L1 cache reference | ~1 ns | 1 second |
| Main memory reference | ~100 ns | 2 minutes |
| SSD random read | ~16 µs | 6 hours |
| Same-DC round trip | ~0.5 ms | 11 days |
| Cross-continent round trip | ~150 ms | 9 years |

### Back-of-Envelope Cheat Sheet

```
1 day  = 86,400 seconds ≈ 100k seconds
1 month ≈ 2.5M seconds

100M requests/day ÷ 100k = 1,000 req/sec (RPS)
Peak is typically 3-5× average → design for 3,000-5,000 RPS

Storage: 1B rows × 1KB = 1TB
Bandwidth: 10M users × 1MB/day = 10TB/day
```

### 🌍 Real-World Scenario: Twitter's Back-of-Envelope

```
300M DAU, 100M tweets/day
→ ~1,200 tweet writes/sec average, ~6,000 peak
→ 300M users × 200 followers each = read fan-out of 60B timeline inserts/day
→ This is 700,000 inserts/sec — can't be done synchronously!
→ Decision: pre-compute timelines async for normal users; pull for celebrity tweets (hybrid fan-out)
```

### 📐 Throughput vs Latency vs Tail

- **Latency**: time per single request
- **Throughput**: requests per second
- **p99 tail**: the worst 1% of requests — what SLAs and real users feel
- **Fan-out multiplication**: in a service that calls 50 downstream services, if each has 1% error rate, your overall error rate approaches `1 - 0.99^50 ≈ 40%`

In [ ]:
"""Back-of-envelope calculator — run this for any system design estimate."""

def back_of_envelope(
    dau: int,
    actions_per_user_per_day: float,
    avg_payload_bytes: int,
    storage_per_record_bytes: int,
    peak_multiplier: float = 3.0,
    years: int = 5,
):
    total_actions_per_day = dau * actions_per_user_per_day
    avg_rps = total_actions_per_day / 86_400
    peak_rps = avg_rps * peak_multiplier
    bandwidth_gbps = (avg_rps * avg_payload_bytes) / 1e9
    storage_5yr_gb = (total_actions_per_day * 365 * years * storage_per_record_bytes) / 1e9

    print("=== Back-of-Envelope Estimate ===")
    print(f"DAU: {dau:,}")
    print(f"Total actions/day: {total_actions_per_day:,.0f}")
    print(f"Average RPS: {avg_rps:,.1f}")
    print(f"Peak RPS ({peak_multiplier}×): {peak_rps:,.1f}")
    print(f"Bandwidth: {bandwidth_gbps:.3f} GB/s")
    print(f"Storage ({years}yr): {storage_5yr_gb:,.1f} GB")

    # Key decision signals
    if peak_rps > 100_000:
        print("⚠️  Sharding likely needed — single DB writer saturates around 50-100k writes/sec")
    if storage_5yr_gb > 10_000:
        print("⚠️  Distributed storage / object store likely needed (>10TB)")
    if bandwidth_gbps > 10:
        print("⚠️  CDN essential — 10+ GB/s origin bandwidth is unsustainable")

# --- Twitter tweet scenario ---
back_of_envelope(
    dau=300_000_000,
    actions_per_user_per_day=0.33,  # ~100M tweets/day
    avg_payload_bytes=500,
    storage_per_record_bytes=1_000,
)
print()
# --- Stripe payments scenario ---
back_of_envelope(
    dau=5_000_000,
    actions_per_user_per_day=1,     # 1 charge/user/day average
    avg_payload_bytes=2_000,
    storage_per_record_bytes=2_000,
)

---
## 3 · CAP Theorem & Consistency Spectrum

### 🧠 Mental Model — *The Two-Switch Analogy*

> **Imagine you have two switches: [Consistency] and [Availability]. During normal operation you can have both. But during a network partition (and they WILL happen), you can only flip ONE switch on. Which one is worth more to your business?**

**WHY**: Network partitions are not hypothetical — cables get cut, data centers lose connectivity, switches fail. Your system will face partitions. If you haven't decided ahead of time which switch to flip, the system will decide for you — usually badly.

**WHAT**: CAP theorem: a distributed store can guarantee at most 2 of: Consistency, Availability, Partition Tolerance. Since P is mandatory, the real choice is **C vs A during a partition**.

**HOW**: This manifests in your storage choice and your code:
- **CP systems** (Postgres, HBase, ZooKeeper): return an error rather than serve stale data
- **AP systems** (Cassandra, DynamoDB, Riak): serve potentially stale data rather than return an error

**WHEN**: Choose per data type, not per system:
- Bank balance → CP (staleness = double-spend, lawsuit)
- Shopping cart → AP (staleness = minor annoyance, fixable)
- Product catalog → AP (1-minute-old price won't hurt anyone)
- User session → AP (acceptable to re-login)
- Order record → CP (must be exactly right)

### The Consistency Spectrum (from strong to weak)

```
STRONG ←————————————————————————————→ WEAK

Linearizable  Serializable  Causal  Read-your-writes  Eventual
    |              |           |           |              |
  Highest        ACID         |      User sees their    Lowest
  latency       (DB tx)    Happens-   own writes      latency
  cost                     before                     cost
```

### PACELC — The More Useful Lens

> **Even when there is NO partition, you trade Latency vs Consistency.** Strong consistency requires coordination (quorum reads, locking) which costs latency. This is the more common daily trade-off.

| Store | Partition behavior | Else behavior |
|---|---|---|
| PostgreSQL | CP | Latency (synchronous) |
| Cassandra (default) | AP | Latency (weak) |
| DynamoDB (strong) | CP | Consistency |
| Redis (no persistence) | AP | Latency |
| Kafka (min.insync.replicas=2) | CP | Consistency |

### 🌍 Real-World: How GitHub's Outage Demonstrated CAP

In 2018, GitHub experienced a network partition between their primary and secondary data centers. Their MySQL replication lost sync. They chose AP (serve reads from potentially stale secondary) rather than CP (take writes offline). This led to ~2 hours where Git pushes from the secondary saw stale data. Post-mortem decision: implement Orchestrator for automated failover with CP semantics for writes.

### 🌍 Real-World: Stripe's Idempotency — CP for Payments

Stripe deliberately chose CP for charge operations. Their `idempotency_key` table is stored in Postgres with a `UNIQUE` constraint — if two concurrent requests try to insert the same key, one gets a unique violation and retries. This guarantees exactly-once charge execution at the cost of latency on concurrent retries.

In [ ]:
"""
Consistency spectrum simulation.
Shows how strong vs eventual consistency behaves under network delay.
"""
from __future__ import annotations
import time
import threading
import random


class StrongConsistencyStore:
    """Linearizable: every read sees the latest write. Coordinated via lock."""
    def __init__(self):
        self._data: dict = {}
        self._lock = threading.Lock()

    def write(self, key: str, value):
        with self._lock:           # coordinate all writers — latency cost
            self._data[key] = value

    def read(self, key: str):
        with self._lock:           # must also lock reads for linearizability
            return self._data.get(key)


class EventualConsistencyStore:
    """AP: writes propagate asynchronously; reads may be stale."""
    def __init__(self, replicas: int = 3, propagation_delay_ms: float = 50):
        self._replicas: list[dict] = [{} for _ in range(replicas)]
        self._delay = propagation_delay_ms / 1000

    def write(self, key: str, value):
        # Write to replica 0 immediately, propagate async
        self._replicas[0][key] = value
        for i in range(1, len(self._replicas)):
            def propagate(r=i, v=value):
                time.sleep(self._delay + random.uniform(0, self._delay))
                self._replicas[r][key] = v
            threading.Thread(target=propagate, daemon=True).start()

    def read(self, key: str, replica: int = None):
        # Random replica — may be stale
        r = random.randint(0, len(self._replicas) - 1) if replica is None else replica
        return self._replicas[r].get(key), r


# Demonstrate eventual consistency staleness window
store = EventualConsistencyStore(replicas=3, propagation_delay_ms=100)
store.write("balance", 1000)

print("Immediately after write (staleness window open):")
for _ in range(5):
    val, replica = store.read("balance")
    print(f"  Replica {replica}: {val}")

time.sleep(0.3)  # wait for propagation
print("\nAfter propagation (all replicas converged):")
for r in range(3):
    val, _ = store.read("balance", replica=r)
    print(f"  Replica {r}: {val}")

---
## 4 · Load Balancing & Consistent Hashing

### 🧠 Mental Model — *The Switchboard Operator*

> **A load balancer is a switchboard operator routing calls to available agents. The strategy determines fairness and stickiness. Consistent hashing is a smarter routing table that doesn't need to be rewritten when an agent joins or leaves.**

**WHY**: Without a load balancer, one server drowns while others idle. Without consistent hashing, adding one cache node remaps all keys — a cache miss storm at the worst possible time (during growth/recovery).

**WHAT**:
- **Round-robin**: requests cycle through servers in order. Simple, ignores capacity.
- **Weighted**: servers get proportional traffic based on capacity. Good for heterogeneous fleets.
- **Least-connections**: requests go to the server with fewest active connections. Best for variable request durations.
- **Key-hash / IP-hash**: same key always routes to same server. Enables server-side caching, session affinity.
- **Consistent hashing**: key maps to server via a ring; adding/removing a server only remaps ~1/N keys.

**HOW**: The consistent hash ring:
```
1. Hash both servers AND keys into a circular [0, 2^32) space
2. A key maps to the first server clockwise from its position
3. Remove server S2: its keys fall to S3 (next clockwise) — only S2's arc is affected
4. Virtual nodes: each physical server has 150+ points on the ring → even distribution
```

**WHEN**:
- Stateless apps → round-robin or least-connections (simple, no stickiness needed)
- Distributed cache (Memcached/Redis cluster) → consistent hashing (minimize cache miss on topology change)
- Session-heavy apps → IP-hash or sticky sessions (accept load imbalance for simplicity)
- Database shard routing → consistent hashing with virtual nodes

### 🌍 Where This Is Seen in Frameworks & Systems

| System | Strategy used | Why |
|---|---|---|
| **Nginx** `upstream` | round-robin (default), least_conn, ip_hash | General-purpose HTTP LB |
| **AWS ALB** | Round-robin per target group | Stateless microservices |
| **Redis Cluster** | Hash slots (16384 slots, consistent-hash-like) | Cache key routing |
| **Memcached** (libmemcached) | Consistent hashing | Minimize cache miss on node add/remove |
| **Cassandra** | Consistent hashing with virtual nodes (vnodes) | Data partitioning across cluster |
| **DynamoDB** | Consistent hashing | Partition key → partition mapping |
| **Envoy Proxy** | Round-robin, least-request, ring-hash | Service mesh LB |
| **gRPC** client-side | Round-robin via `grpc.experimental.ChannelZ` | Client-side LB for gRPC services |

### ❌ Before: Naive Hash — The Reshuffling Disaster

In [ ]:
# ❌ BEFORE: hash(key) % N — everything remaps when N changes

def naive_route(key: str, num_servers: int) -> int:
    return hash(key) % num_servers

keys = [f"user:{i}" for i in range(1000)]

# Assign keys to 4 servers
mapping_4 = {k: naive_route(k, 4) for k in keys}

# Add 1 server → now 5
mapping_5 = {k: naive_route(k, 5) for k in keys}

# Count how many keys moved
moved = sum(1 for k in keys if mapping_4[k] != mapping_5[k])
print(f"❌ Naive hash: {moved}/{len(keys)} keys remapped when adding 1 server ({moved/len(keys)*100:.0f}%)")
# → ~800 of 1000 keys move — that's 80% cache miss storm!

In [ ]:
# ✅ AFTER: Consistent hashing — only ~1/N keys move

import bisect
import hashlib


def _hash(s: str) -> int:
    return int(hashlib.md5(s.encode()).hexdigest(), 16)


class ConsistentHashRing:
    """
    Mental model: a clock-face where servers sit at positions.
    A key goes to the first server clockwise from its own position.
    Virtual nodes (replicas) prevent hot spots when servers are unevenly placed.
    """
    def __init__(self, virtual_nodes: int = 150):
        self._virtual_nodes = virtual_nodes
        self._ring: dict[int, str] = {}   # hash → server name
        self._sorted: list[int] = []      # sorted ring positions

    def add_server(self, server: str) -> None:
        for i in range(self._virtual_nodes):
            h = _hash(f"{server}#{i}")
            self._ring[h] = server
        self._sorted = sorted(self._ring)

    def remove_server(self, server: str) -> None:
        for i in range(self._virtual_nodes):
            h = _hash(f"{server}#{i}")
            del self._ring[h]
        self._sorted = sorted(self._ring)

    def get_server(self, key: str) -> str | None:
        if not self._ring:
            return None
        h = _hash(key)
        idx = bisect.bisect(self._sorted, h) % len(self._sorted)
        return self._ring[self._sorted[idx]]


# Assign 1000 keys with 4 servers
ring4 = ConsistentHashRing(virtual_nodes=150)
for s in ["s1", "s2", "s3", "s4"]:
    ring4.add_server(s)

keys = [f"user:{i}" for i in range(1000)]
mapping_ch4 = {k: ring4.get_server(k) for k in keys}

# Add server s5
ring5 = ConsistentHashRing(virtual_nodes=150)
for s in ["s1", "s2", "s3", "s4", "s5"]:
    ring5.add_server(s)
mapping_ch5 = {k: ring5.get_server(k) for k in keys}

moved_ch = sum(1 for k in keys if mapping_ch4[k] != mapping_ch5[k])
print(f"✅ Consistent hash: {moved_ch}/{len(keys)} keys remapped when adding 1 server ({moved_ch/len(keys)*100:.0f}%)")
# → ~200 of 1000 keys move — only those belonging to the new server's arc

# Load distribution (should be roughly even due to virtual nodes)
from collections import Counter
dist = Counter(mapping_ch5.values())
print("\nLoad distribution across 5 servers:")
for s, count in sorted(dist.items()):
    bar = '█' * (count // 5)
    print(f"  {s}: {count:3d} keys {bar}")

---
## 5 · Caching Deep Dive — All Layers

### 🧠 Mental Model — *The Grocery Store*

> **Imagine you need butter. The pantry (in-memory cache) is 3 seconds away. The local store (Redis) is 10 minutes. The warehouse (database) is 2 hours. You stock the pantry with what you use most, refresh weekly, and go to the warehouse only when the pantry and store run dry. Cache hit rate is how often you find butter in the pantry.**

**WHY**: A database read under load is 1-10ms and uses CPU+disk. A Redis read is 0.1-0.5ms. A memory read is <1µs. Caching shifts the access tier for hot data, dramatically increasing throughput at the same hardware cost.

**WHAT**: A hierarchy of stores, each progressively closer to the user:
```
Browser Cache → CDN Edge → API Gateway Cache → Application Cache (Redis) → DB Buffer Pool → DB
     Fastest ←————————————————————————————————————————————————————————————————→ Slowest
```

**HOW**: Three write strategies with different trade-offs:

| Strategy | Write path | Consistency | Use when |
|---|---|---|---|
| **Cache-aside** (lazy) | Write DB only; cache filled on next read miss | Eventual | Most reads, moderate writes |
| **Write-through** | Write cache AND DB synchronously | Strong | Read-heavy, can't tolerate stale |
| **Write-back** (write-behind) | Write cache only; flush to DB async | Weakest | Write-heavy, can tolerate some loss |

**WHEN**:
- Use cache-aside for most general-purpose caching (user profiles, product data)
- Use write-through for financial balances where staleness is dangerous
- Use write-back for metrics/counters where a few lost updates are tolerable

### 🌍 Real-World: How Netflix Uses Caching

Netflix operates **EVCache** (a Memcached wrapper): geographically distributed, stores user-session and recommendation data. ~99% of their reads hit EVCache. They also use **local in-process caches** (Guava/Caffeine) to avoid even the EVCache network hop for static metadata. Two-level caching: L1 (local JVM heap) → L2 (EVCache cluster) → Cassandra.

### The Three Hard Problems of Caching

**1. Cache Stampede (Thundering Herd)**
> A hot key expires. 10,000 concurrent requests all miss the cache simultaneously and all hit the database. The database buckles under the load.
> Fix: **Probabilistic early expiry** or **mutex/semaphore** (only one request recomputes; others wait or serve stale)

**2. Cache Invalidation**
> You update a product price in the DB. The old price is still in cache for 1 hour. Customers see wrong price.
> Fix: **Event-driven invalidation** (publish a `ProductUpdated` event → consumers delete/update cache key), or **shorter TTL** (accept bounded staleness)

**3. Cache Penetration**
> Attackers query for IDs that don't exist (user:999999). Every request misses cache and hits DB. DB exhausted.
> Fix: **Bloom filter** (cache negative lookups) or **null caching** (store a sentinel value for missing keys)

### 🌍 Where This Is Seen in Frameworks

| Framework/Tool | Caching pattern | Notes |
|---|---|---|
| **Django** `cache_page` | Full-page cache-aside | Built on Redis/Memcached backend |
| **FastAPI** with `aiocache` | Function-level cache-aside | Async cache decorators |
| **Spring Cache** `@Cacheable` | Cache-aside declarative | Backed by Redis or Caffeine |
| **Rails** `fetch` | Cache-aside with block | Block only executed on miss |
| **Nginx** `proxy_cache` | CDN-like response caching | Upstream cache hits bypass app |
| **Redis** `SET EX NX` | Distributed mutex | Stampede prevention |
| **Celery** result backend | Write-through | Task results stored on completion |

In [ ]:
"""
All three write strategies + stampede prevention — runnable.

❌ BEFORE: No caching — every read hits the DB.
✅ AFTER: Cache-aside + mutex stampede prevention.
"""
from __future__ import annotations
import time


# ============================================================
# ❌ BEFORE: Direct DB access, no caching
# ============================================================
_DB = {"product:1": {"name": "Widget", "price": 9.99}}
_db_calls_before = 0

def get_product_before(product_id: str):
    global _db_calls_before
    _db_calls_before += 1
    time.sleep(0.005)  # simulate 5ms DB read
    return _DB.get(product_id)

# Simulate 100 concurrent requests for the same product
results_before = []
threads = [threading.Thread(target=lambda: results_before.append(get_product_before("product:1"))) for _ in range(10)]
[t.start() for t in threads]
[t.join() for t in threads]
print(f"❌ Before: {_db_calls_before} DB calls for 10 requests to same product")


# ============================================================
# ✅ AFTER: Cache-aside with stampede prevention (mutex)
# ============================================================
_cache: dict = {}
_cache_locks: dict[str, threading.Lock] = {}
_db_calls_after = 0

def get_product_after(product_id: str, ttl: float = 60.0):
    """Cache-aside: check cache → on miss, acquire key-specific lock → load DB → fill cache."""
    global _db_calls_after

    # Fast path: cache hit (no lock needed)
    entry = _cache.get(product_id)
    if entry and time.time() < entry["expires"]:
        return entry["value"]

    # Slow path: key-level lock prevents stampede
    lock = _cache_locks.setdefault(product_id, threading.Lock())
    with lock:
        # Double-check: another thread may have filled cache while we waited
        entry = _cache.get(product_id)
        if entry and time.time() < entry["expires"]:
            return entry["value"]

        # This thread wins the DB read
        _db_calls_after += 1
        time.sleep(0.005)  # simulate 5ms DB read
        value = _DB.get(product_id)
        _cache[product_id] = {"value": value, "expires": time.time() + ttl}
        return value

results_after = []
threads2 = [threading.Thread(target=lambda: results_after.append(get_product_after("product:1"))) for _ in range(10)]
[t.start() for t in threads2]
[t.join() for t in threads2]
print(f"✅ After:  {_db_calls_after} DB call(s) for 10 requests to same product (mutex coalesced the stampede)")
print(f"   All results correct: {all(r == _DB['product:1'] for r in results_after)}")

---
## 6 · Data Partitioning & Replication

### 🧠 Mental Model — *The Library System*

> **Replication is making copies of books for multiple branches — if one branch burns down, you still have the book. Partitioning (sharding) is deciding which books go to which branch — you can serve more readers in parallel, but you can only borrow a book from the branch that has it.**

**WHY**:
- **Replication**: fault tolerance (one server dies → others take over) and read scale (route reads to replicas)
- **Partitioning**: write scale (each shard handles only its subset) and storage scale (data distributed across machines)

**WHAT**:

**Replication topologies:**
```
Single-leader: PRIMARY ──writes──> (replicates async) ──> REPLICA1, REPLICA2
               reads can hit any replica

Multi-leader:  Primary A ←──sync──→ Primary B (conflict resolution needed)
               used for geo-distributed writes (CockroachDB, Vitess)

Leaderless:    Write to W of N replicas; read from R of N (Cassandra, Dynamo)
               R + W > N = quorum → strong consistency
```

**Sharding strategies:**
```
Range sharding:  user_id 0-999 → shard1, 1000-1999 → shard2, ...
  Pros: range queries work. Cons: hotspots (all new users → last shard)

Hash sharding:   shard = hash(user_id) % num_shards
  Pros: even distribution. Cons: range queries require scatter-gather

Directory sharding: lookup service maps key → shard
  Pros: flexible. Cons: lookup service = single point of failure

Geo sharding:    EU users → EU shard, US users → US shard
  Pros: data locality, GDPR compliance. Cons: cross-region queries hard
```

**HOW** — The Shard Key Is Everything:
- Choose a key with **high cardinality** (many distinct values → even spread)
- Choose a key that **matches your access patterns** (queries on `user_id` → shard by `user_id`)
- Avoid keys that **create hotspots** (timestamp → all writes hit the latest shard)

**WHEN**:
- Start with replication only (read replicas) — this handles most read scale
- Add sharding when you've exhausted vertical scaling for writes (~50-100k writes/sec on a single primary)
- Sharding is painful to add later — factor it into schema early

### 🌍 Real-World Sharding Examples

| Company | Data | Shard key | Why |
|---|---|---|---|
| **Instagram** | Photos | `user_id` | Most queries are per-user |
| **Stripe** | Transactions | `merchant_id` | Merchant-level queries dominate |
| **Discord** | Messages | `channel_id` % N | Channel access is the unit |
| **MongoDB Atlas** | Varies | Configurable | Zone sharding for geo |
| **YouTube** | Videos | `video_id` | Video metadata/view access |

### 🌍 Where This Is Seen in Frameworks

| Framework/Tool | Replication/Sharding feature | Notes |
|---|---|---|
| **SQLAlchemy** | `read_preference`, `bind_engine` | Manual shard routing via session binders |
| **Django** | `DATABASE_ROUTERS` | Route reads to replicas declaratively |
| **Vitess** | Automatic horizontal sharding for MySQL | Used by GitHub, Slack, Pinterest |
| **Cassandra** | Built-in consistent-hash partitioning | Tunable consistency (R+W>N) |
| **MongoDB** | `mongos` shard router | Chunk-based range sharding |
| **PostgreSQL** | `pg_partman`, Citus | Declarative range/hash partitioning |

### ⚠️ Replication Lag — The Silent Danger

In [ ]:
"""
Replication lag simulation + read-your-own-writes fix.

❌ BEFORE: Read from replica immediately after write → stale data (user doesn't see their own update)
✅ AFTER:  Route user's own recent writes to primary for a short window
"""
from __future__ import annotations
import time
import threading


class PrimaryReplicaDB:
    def __init__(self, replication_lag_ms: float = 100):
        self._primary: dict = {}
        self._replica: dict = {}
        self._lag = replication_lag_ms / 1000

    def write(self, key: str, value) -> None:
        self._primary[key] = value
        # Async replication with lag
        def replicate():
            time.sleep(self._lag)
            self._replica[key] = value
        threading.Thread(target=replicate, daemon=True).start()

    def read_primary(self, key: str):
        return self._primary.get(key)

    def read_replica(self, key: str):
        return self._replica.get(key)   # may be stale!


class ReadYourWritesSession:
    """
    ✅ After: Track recent writes per user.
    For a short window after a write, route that user's reads to primary.
    This ensures users always see their own updates (read-your-writes guarantee).
    """
    def __init__(self, db: PrimaryReplicaDB, primary_window_s: float = 5.0):
        self._db = db
        self._window = primary_window_s
        self._user_wrote_at: dict[str, float] = {}

    def write(self, user_id: str, key: str, value) -> None:
        self._db.write(key, value)
        self._user_wrote_at[user_id] = time.time()

    def read(self, user_id: str, key: str):
        wrote_at = self._user_wrote_at.get(user_id, 0)
        if time.time() - wrote_at < self._window:
            # Route to primary — this user recently wrote
            return self._db.read_primary(key), "primary"
        return self._db.read_replica(key), "replica"


db = PrimaryReplicaDB(replication_lag_ms=100)
session = ReadYourWritesSession(db)

# ❌ Before: write then immediately read from replica
db.write("user:42:name", "Alice")
stale_read = db.read_replica("user:42:name")
print(f"❌ Before (read replica immediately): '{stale_read}' (None = stale, replication lag)")

# ✅ After: session routes to primary within window
session.write("user_42", "user:42:name", "Alice")
value, source = session.read("user_42", "user:42:name")
print(f"✅ After  (read-your-writes session): '{value}' from {source}")

time.sleep(0.15)  # wait for replication
# Another user (no recent write) reads from replica — now caught up
value2, source2 = session.read("other_user", "user:42:name")
print(f"   Other user after lag: '{value2}' from {source2}")

---
## 7 · Reliability Patterns — The Production Survival Kit

### 🧠 Mental Model — *The Electrical Grid*

> **A city's power grid uses circuit breakers, load shedding, and redundant lines so that a failure in one neighborhood doesn't cascade into a blackout. Your distributed system needs the same: circuit breakers stop cascading failures, rate limiting is load shedding, and retries with backoff are the redundant paths.**

**WHY**: In a distributed system, failures are not exceptional — they are routine. A server will crash, a network will partition, a downstream API will go slow. Without reliability patterns, one slow service cascades into a total outage (all threads block waiting for it → your service goes down too).

### Pattern 1: Circuit Breaker

**What**: Three-state state machine: CLOSED (normal) → OPEN (failing, fail-fast) → HALF-OPEN (probe for recovery)

**When**: Any synchronous call to a downstream service that could go slow or fail

**Where seen**: Netflix Hystrix (origin), Resilience4j, Polly (.NET), `circuitbreaker` in Go, `pybreaker` in Python, AWS App Mesh, Envoy proxy

### Pattern 2: Retry with Exponential Backoff + Jitter

**What**: Retry failed requests, but with increasing delays + random jitter to prevent synchronized retries from overwhelming the recovering service

**When**: Idempotent operations against services that may have transient failures

**Where seen**: AWS SDK retry logic, `tenacity` Python library, gRPC retry policy, Kubernetes job backoff

### Pattern 3: Idempotency Keys

**What**: Client sends a unique key with each request; server stores result keyed by it; retries replay the stored result rather than re-executing the side effect

**When**: Any state-mutating operation (payments, order creation, email sending) where retries must not double-execute

**Where seen**: Stripe API `Idempotency-Key` header, Shopify webhook deduplication, AWS SQS FIFO message deduplication IDs

### Pattern 4: Rate Limiting

**What**: Throttle request rate per client/tenant to protect the system from overload

**When**: Any public-facing API; any expensive internal operation

**Where seen**: GitHub API rate limits (5000/hour), AWS API Gateway throttling, FastAPI `slowapi`, Nginx `limit_req_zone`, Redis `INCR` + TTL

### Pattern 5: Bulkhead

**What**: Isolate resources (thread pools, connection pools) per downstream service so one slow dependency can't exhaust all connections

**When**: Services calling multiple downstream dependencies

**Where seen**: Hystrix thread pool isolation, Gunicorn worker pools, asyncio semaphores

### 🌍 Real-World: Amazon's Dependency Limit Rule

Amazon discovered that a single slow dependency cascaded to take down their entire checkout flow. Response: every service must have an **explicit timeout** for every downstream call and must **fail fast** (return a degraded response or error) rather than wait indefinitely. The Bulkhead pattern ensures a slow inventory service can't block the payment service's thread pool.

In [ ]:
"""
Circuit Breaker + Retry with Backoff + Idempotency — all together.

❌ BEFORE: Direct calls, no retry logic, no deduplication
✅ AFTER:  Circuit breaker protects, retry with jitter recovers, idempotency prevents double-charge
"""
from __future__ import annotations
import time
from enum import Enum
from typing import Any
from collections.abc import Callable


# ─── Circuit Breaker ─────────────────────────────────────────────────────────
class CircuitState(Enum):
    CLOSED = "closed"           # Normal: requests pass through
    OPEN = "open"               # Failing: fail-fast, no real calls
    HALF_OPEN = "half_open"     # Probing: test if service recovered

class CircuitBreaker:
    """
    Mental model: a physical circuit breaker in your home.
    Too much current (failures) → it trips OPEN → you're protected from the surge.
    After a cool-down, try one probe (HALF-OPEN).
    Success → CLOSED again. Failure → back to OPEN.
    """
    def __init__(self, failure_threshold: int = 3, recovery_timeout_s: float = 5.0):
        self.state = CircuitState.CLOSED
        self._failures = 0
        self._threshold = failure_threshold
        self._opened_at: float = 0
        self._timeout = recovery_timeout_s

    def call(self, fn: Callable, *args, **kwargs) -> Any:
        if self.state == CircuitState.OPEN:
            if time.time() - self._opened_at >= self._timeout:
                self.state = CircuitState.HALF_OPEN
                print("  [CB] Probing recovery (HALF-OPEN)")
            else:
                raise RuntimeError("Circuit OPEN — fail fast")  # no real call

        try:
            result = fn(*args, **kwargs)
            self._on_success()
            return result
        except Exception:
            self._on_failure()
            raise

    def _on_success(self):
        self._failures = 0
        if self.state != CircuitState.CLOSED:
            print("  [CB] Recovered → CLOSED")
        self.state = CircuitState.CLOSED

    def _on_failure(self):
        self._failures += 1
        if self._failures >= self._threshold:
            self.state = CircuitState.OPEN
            self._opened_at = time.time()
            print(f"  [CB] {self._failures} failures → OPEN (fail-fast mode)")


# ─── Retry with Exponential Backoff + Jitter ─────────────────────────────────
def retry_with_backoff(
    fn: Callable,
    max_attempts: int = 3,
    base_delay_s: float = 0.1,
    max_delay_s: float = 2.0,
    *args, **kwargs
) -> Any:
    """
    Full jitter (sleep = random * min(cap, base * 2^attempt)).
    Jitter prevents synchronized retries from creating a second wave of overload.
    """
    for attempt in range(max_attempts):
        try:
            return fn(*args, **kwargs)
        except RuntimeError as e:
            if attempt == max_attempts - 1:
                raise
            cap = min(max_delay_s, base_delay_s * (2 ** attempt))
            sleep = random.uniform(0, cap)  # full jitter
            print(f"  [Retry] attempt {attempt+1} failed: {e}. Sleeping {sleep:.3f}s")
            time.sleep(sleep)


# ─── Idempotency Store ────────────────────────────────────────────────────────
_idempotency_store: dict[str, dict] = {}
_charge_count = 0

def charge_card_idempotent(idempotency_key: str, amount: float) -> dict:
    """✅ After: replay stored result on retry; only charge once."""
    if idempotency_key in _idempotency_store:
        print(f"  [Idempotency] Replaying stored result for key={idempotency_key}")
        return _idempotency_store[idempotency_key]

    global _charge_count
    _charge_count += 1
    result = {"charge_id": f"txn-{_charge_count}", "amount": amount, "status": "ok"}
    _idempotency_store[idempotency_key] = result  # persist BEFORE returning
    return result


# ─── Demo ─────────────────────────────────────────────────────────────────────
print("=== Circuit Breaker Demo ===")
call_count = 0

def flaky_service():
    global call_count
    call_count += 1
    if call_count <= 3:
        raise RuntimeError(f"Service down (call {call_count})")
    return f"Success (call {call_count})"

cb = CircuitBreaker(failure_threshold=3, recovery_timeout_s=0.1)

for i in range(7):
    try:
        result = cb.call(flaky_service)
        print(f"  Call {i+1}: {result} [{cb.state.value}]")
    except RuntimeError as e:
        print(f"  Call {i+1}: ✗ {e} [{cb.state.value}]")
    if i == 4:
        time.sleep(0.15)  # let circuit cool down

print("\n=== Idempotency Demo ===")
key = "payment-uuid-abc123"
r1 = charge_card_idempotent(key, 99.99)
r2 = charge_card_idempotent(key, 99.99)  # retry with same key
r3 = charge_card_idempotent(key, 99.99)  # another retry
print(f"3 calls with same key → {_charge_count} actual charge(s): {r1['charge_id']}")
assert r1 == r2 == r3, "Idempotency violation!"
print("✅ All three responses identical — exactly-once effect")

---
## 8 · API Design at Scale — REST · gRPC · GraphQL

### 🧠 Mental Model — *Three Languages for the Same Conversation*

> **REST is English — everyone speaks it, it's flexible, verbose. gRPC is Latin — formal, compact, precise, understood by specialists. GraphQL is a questionnaire — you specify exactly what you want and get exactly that, nothing more, nothing less.**

| Dimension | REST | gRPC | GraphQL |
|---|---|---|---|
| **Protocol** | HTTP/1.1 (text) | HTTP/2 (binary) | HTTP/1.1 or 2 |
| **Schema** | Optional (OpenAPI) | Mandatory (`.proto`) | Mandatory (SDL) |
| **Payload** | JSON (verbose) | Protobuf (compact) | JSON |
| **Streaming** | SSE / WebSocket | Native bi-directional | Subscriptions |
| **N+1 problem** | Common | Common | Built-in problem; fix = DataLoader |
| **Caching** | HTTP cache (easy) | Harder (POST) | Harder (variable queries) |
| **Code gen** | Optional | Mandatory (protoc) | Optional |
| **Best for** | Public APIs, CRUD | Internal high-perf services | Client-driven data fetching |

### 🌍 Real-World Usage
- **Twitter API v2**: REST for external devs (well-understood, documented)
- **Google internal (Stubby → gRPC)**: ~10B inter-service gRPC calls/second
- **GitHub API v4**: GraphQL (developers querying complex repo graphs efficiently)
- **Netflix**: gRPC between internal microservices (bandwidth/latency savings)
- **Shopify**: GraphQL (merchants have wildly different data needs → client-driven)

### The N+1 Problem — GraphQL's Achilles Heel

```graphql
# Query: list 100 orders, each with user name
{ orders { id user { name } } }

# Naive resolver: 1 DB query for orders + 100 queries for user per order = 101 queries!
# Fix: DataLoader (batches + deduplicates: 1 + 1 = 2 queries total)
```

### 🌍 Where This Is Seen in Frameworks

| Pattern | Framework/Tool |
|---|---|
| REST routing | FastAPI, Django REST Framework, Flask-RESTX |
| gRPC codegen | `protoc` + `grpcio-tools`, `betterproto` |
| GraphQL server | Strawberry (FastAPI), Ariadne, Graphene |
| N+1 fix | `strawberry-graphql-django` DataLoader, `aiodataloader` |
| API versioning | URL prefix (`/v1/`), header (`Accept: v2`), content negotiation |
| Rate limiting | `slowapi` (FastAPI), Django Ratelimit, Kong gateway |

In [ ]:
"""
N+1 problem simulation and DataLoader-style batching fix.

❌ BEFORE: Resolve each user separately → N+1 DB queries
✅ AFTER:  DataLoader batches all user IDs into a single query
"""
from __future__ import annotations


# Simulated DB
_ORDERS_DB = [{"id": i, "user_id": i % 5} for i in range(20)]
_USERS_DB = {i: {"id": i, "name": f"User {i}"} for i in range(5)}
_query_count = 0

def db_get_user(user_id: int) -> dict:
    """Simulates one DB round-trip per call."""
    global _query_count
    _query_count += 1
    return _USERS_DB[user_id]

def db_get_users_batch(user_ids: list[int]) -> dict[int, dict]:
    """One DB round-trip for all user IDs (WHERE id IN (...))."""
    global _query_count
    _query_count += 1
    return {uid: _USERS_DB[uid] for uid in user_ids if uid in _USERS_DB}


# ❌ BEFORE: N+1 — resolver calls db_get_user once per order
_query_count = 0
result_before = []
for order in _ORDERS_DB:
    user = db_get_user(order["user_id"])  # 1 query per order!
    result_before.append({**order, "user_name": user["name"]})

print(f"❌ N+1 Before: {_query_count} DB queries for {len(_ORDERS_DB)} orders")


# ✅ AFTER: DataLoader pattern — collect all IDs, batch-fetch, distribute
class DataLoader:
    """
    Mental model: a shipping batch scheduler.
    Instead of sending a truck for each package (N queries), it waits to fill
    the truck (collect all IDs) then sends one trip (1 batch query).
    """
    def __init__(self, batch_fn):
        self._batch_fn = batch_fn
        self._queue: list = []
        self._cache: dict = {}

    def load(self, key) -> object:
        """Schedule key for loading (returns a future-like sentinel)."""
        if key not in self._cache:
            self._queue.append(key)
        return key  # simplified; real DataLoader returns a Promise

    def dispatch_all(self) -> dict:
        """Batch-fetch all queued keys in one call."""
        if self._queue:
            unique_keys = list(dict.fromkeys(self._queue))
            results = self._batch_fn(unique_keys)
            self._cache.update(results)
            self._queue.clear()
        return self._cache


_query_count = 0
user_loader = DataLoader(db_get_users_batch)

# Phase 1: Register all needed user IDs (no DB calls yet)
order_user_keys = [user_loader.load(order["user_id"]) for order in _ORDERS_DB]

# Phase 2: One batch fetch for all unique user IDs
user_map = user_loader.dispatch_all()

# Phase 3: Assemble results
result_after = [
    {**order, "user_name": user_map[order["user_id"]]["name"]}
    for order in _ORDERS_DB
]

print(f"✅ DataLoader After: {_query_count} DB query for {len(_ORDERS_DB)} orders")
print(f"   Same results: {[r['user_name'] for r in result_before] == [r['user_name'] for r in result_after]}")

---
## 9 · Monolith → Microservices Spectrum

### 🧠 Mental Model — *The Restaurant Kitchen*

> **A food truck (monolith): one chef does everything — fast, simple, tight coordination. A multi-station restaurant kitchen (microservices): separate stations for grill, salads, desserts — each scales independently, but the head chef (API gateway) must coordinate, and a slow dessert station (downstream service) can hold up the entire table.**

**WHY**: Monoliths become painful when:
1. Build + deploy time grows (changing one function requires deploying 500k lines)
2. Teams step on each other (10 teams, 1 repo, 1 release train)
3. Scaling is all-or-nothing (can't scale just the video encoding without scaling everything)

**WHAT**: The spectrum:
```
Monolith → Modular Monolith → Macro-services → Microservices → Nano-services (anti-pattern)
```

**HOW**: The Strangler Fig pattern for safe migration:
```
1. Put a proxy/gateway in front of the monolith
2. Extract one high-value, stable service (e.g., authentication)
3. Route that traffic to the new service via the proxy
4. Verify, stabilize, extract next service
5. Old monolith shrinks until it's gone (or you stop — both are fine!)
```

**WHEN**: Start microservices when:
- You have >3 teams and they're consistently blocked on shared deployments
- You have clear, stable service boundaries (don't split prematurely)
- You have operational maturity (CI/CD, observability, service mesh)

### ⚠️ The Distributed Monolith Anti-Pattern

> If your "microservices" share a database, must be deployed together, or chain 10 synchronous calls for every request — you've built a distributed monolith. You have all the pain of distribution (network calls, distributed transactions, partial failures) with none of the independence.

**Signs you have a distributed monolith:**
- Services cannot be deployed independently
- Services share a single database schema
- A change in Service A requires changes in Service B and C
- 10+ synchronous hops per user request

### 🌍 Real-World Migration Stories

| Company | Journey | Key insight |
|---|---|---|
| **Amazon** | Rails monolith → microservices (2002) | Bezos memo: all teams must expose APIs; no backdoor DB access |
| **Netflix** | DVD monolith → microservices (2007-2012) | Cloud outage triggered microservices journey; ~700 services today |
| **Shopify** | Rails monolith → modular monolith (not microservices!) | Chose modular monolith over microservices; pods concept |
| **Stack Overflow** | Microservices → monolith (reverse!) | Found monolith simpler and faster for their scale |
| **Segment** | Microservices → monolith | 130 microservices → operational nightmare → merged back |

### 🌍 Where This Is Seen in Frameworks

| Pattern | Framework/Tool |
|---|---|
| API Gateway | FastAPI (custom), Kong, AWS API Gateway, Nginx |
| Service discovery | Consul, Kubernetes DNS, AWS Service Discovery |
| Service mesh | Istio, Linkerd, Envoy |
| Strangler fig routing | Nginx `location` blocks, AWS ALB rules |
| Health checks | FastAPI `/health`, Kubernetes liveness/readiness probes |

In [ ]:
"""
Strangler Fig pattern — safe incremental migration from monolith to microservice.

❌ BEFORE: All traffic goes to monolith (including the new auth service that should be separate)
✅ AFTER:  Gateway routes /auth/* to new service, everything else to monolith
         Safe to roll back by updating the routing table.
"""
from __future__ import annotations


# ❌ BEFORE: monolith handles everything
def legacy_monolith(path: str, method: str, body: dict) -> dict:
    routes = {
        ("/auth/login", "POST"): lambda b: {"token": "legacy-jwt", "source": "monolith"},
        ("/products", "GET"): lambda b: {"products": [], "source": "monolith"},
        ("/orders", "POST"): lambda b: {"order_id": 1, "source": "monolith"},
    }
    handler = routes.get((path, method))
    return handler(body) if handler else {"error": "not found"}


# ✅ AFTER: New auth microservice
def auth_microservice(path: str, method: str, body: dict) -> dict:
    if path == "/auth/login" and method == "POST":
        return {"token": "new-jwt", "source": "auth-service-v2"}
    return {"error": "not found"}


# ✅ AFTER: Strangler Fig gateway — routes based on path prefix
class StranglerFigGateway:
    """
    Mental model: a traffic cop standing at the old building entrance.
    Auth requests are directed to the new auth building.
    Everything else still goes to the old building.
    Neither building knows about the other.
    """
    def __init__(self):
        # prefix → handler (could be URL rewrite in Nginx/ALB)
        self._routes: list[tuple[str, Callable]] = []
        self._fallback: Callable = None

    def add_route(self, prefix: str, handler: Callable) -> None:
        self._routes.append((prefix, handler))

    def set_fallback(self, handler: Callable) -> None:
        self._fallback = handler

    def handle(self, path: str, method: str, body: dict) -> dict:
        for prefix, handler in self._routes:
            if path.startswith(prefix):
                return handler(path, method, body)
        return self._fallback(path, method, body)


gateway = StranglerFigGateway()
gateway.add_route("/auth", auth_microservice)   # extracted service
gateway.set_fallback(legacy_monolith)            # everything else → old monolith

# Test routing
login_result = gateway.handle("/auth/login", "POST", {"user": "alice"})
products_result = gateway.handle("/products", "GET", {})
orders_result = gateway.handle("/orders", "POST", {"items": [1, 2]})

print(f"POST /auth/login → {login_result['source']} ({login_result['token']})")
print(f"GET  /products   → {products_result['source']}")
print(f"POST /orders     → {orders_result['source']}")
print()
print("✅ Auth traffic migrated to new service; rest still on monolith.")
print("   Rollback: remove the /auth route from gateway. No deployment needed.")

---
## 10 · Clean / Hexagonal Architecture — Structure That Survives

### 🧠 Mental Model — *The Onion*

> **Peel an onion from the outside in. The outer layers (Web, Database, Email) are easy to replace — they're just adapters. The inner layers (Use Cases, Domain) contain the business rules that never change. The key rule: inner layers NEVER import outer layers. Dependencies only flow inward.**

**WHY**: When your domain logic imports SQLAlchemy directly, you can't test it without a database. When it imports FastAPI, you can't use it in a CLI. Inversion of dependency lets you test the core logic with simple in-memory fakes, and swap Postgres for MongoDB without touching business rules.

**WHAT**:
```
[Frameworks/Drivers]  ← FastAPI, SQLAlchemy, Redis, Celery
    ↓ depends on
[Interface Adapters]  ← Controllers, Repositories, Serializers
    ↓ depends on
[Use Cases]           ← PlaceOrder, ChargePayment, RegisterUser
    ↓ depends on
[Domain/Entities]     ← Order, Payment, User (pure Python, no imports)
```

**HOW** — Ports & Adapters:
- **Port** = interface that the domain owns and defines (`OrderRepository` abstract class)
- **Adapter** = concrete implementation (`PostgresOrderRepository`, `InMemoryOrderRepository`)
- The domain depends only on the Port; production wires `PostgresOrderRepository`, tests wire `InMemoryOrderRepository`

**WHEN**: Any service with meaningful business logic that you want to test without spinning up a database. Not for simple CRUD APIs — overkill there.

### 🌍 Where This Is Seen in Frameworks

| Pattern | Framework/Tool |
|---|---|
| Repository pattern | Django: `Model.objects` as implicit repo; FastAPI: explicit repo class |
| Dependency injection | FastAPI `Depends()`, Python `inject` library |
| Clean arch for Django | `django-clean-architecture`, custom `services.py` layer |
| Testing with fakes | `unittest.mock`, `pytest-mock`, in-memory SQLite |

In [ ]:
"""
Hexagonal / Clean Architecture — Ports & Adapters pattern.

❌ BEFORE: Use case imports SQLAlchemy directly → untestable without DB
✅ AFTER:  Use case depends on abstract Port → test with in-memory adapter, production with Postgres
"""
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field


# ─── Domain (innermost — zero dependencies) ──────────────────────────────────
@dataclass
class Order:
    order_id: str
    user_id: str
    amount_cents: int
    status: str = "pending"

    def mark_paid(self):
        if self.status != "pending":
            raise ValueError(f"Cannot pay order in state: {self.status}")
        self.status = "paid"


# ─── Port (abstract interface — domain owns this) ─────────────────────────────
class OrderRepository(ABC):
    """Port: The domain defines what it needs. Infrastructure implements it."""
    @abstractmethod
    def save(self, order: Order) -> None: ...

    @abstractmethod
    def find_by_id(self, order_id: str) -> Order | None: ...


class PaymentGateway(ABC):
    @abstractmethod
    def charge(self, user_id: str, amount_cents: int) -> str: ...  # returns charge_id


# ─── Use Case (depends only on ports — no DB, no web framework) ──────────────
class PlaceOrderUseCase:
    """
    Business logic lives here, purely.
    The use case doesn't know if the repo is Postgres or in-memory.
    The use case doesn't know if it's called from FastAPI or a CLI.
    """
    def __init__(self, repo: OrderRepository, payment: PaymentGateway):
        self._repo = repo
        self._payment = payment

    def execute(self, order_id: str, user_id: str, amount_cents: int) -> Order:
        order = Order(order_id=order_id, user_id=user_id, amount_cents=amount_cents)
        charge_id = self._payment.charge(user_id, amount_cents)
        order.mark_paid()
        self._repo.save(order)
        return order


# ─── Adapters (outer ring — infrastructure) ──────────────────────────────────
class InMemoryOrderRepository(OrderRepository):
    """Test adapter — no database needed."""
    def __init__(self):
        self._store: dict[str, Order] = {}

    def save(self, order: Order) -> None:
        self._store[order.order_id] = order

    def find_by_id(self, order_id: str) -> Order | None:
        return self._store.get(order_id)


class FakePaymentGateway(PaymentGateway):
    """Test adapter — no real Stripe call."""
    def __init__(self, should_fail: bool = False):
        self._should_fail = should_fail

    def charge(self, user_id: str, amount_cents: int) -> str:
        if self._should_fail:
            raise RuntimeError("Payment declined")
        return f"fake-charge-{user_id}-{amount_cents}"


# ─── Test (no database, no HTTP, no external services) ───────────────────────
repo = InMemoryOrderRepository()
payment = FakePaymentGateway(should_fail=False)
use_case = PlaceOrderUseCase(repo=repo, payment=payment)

order = use_case.execute("order-001", "user-42", 9999)
assert order.status == "paid"
assert repo.find_by_id("order-001").status == "paid"
print(f"✅ Order {order.order_id} placed and paid — tested with zero infrastructure")

# Test failure path
repo2 = InMemoryOrderRepository()
payment_fail = FakePaymentGateway(should_fail=True)
use_case_fail = PlaceOrderUseCase(repo=repo2, payment=payment_fail)

try:
    use_case_fail.execute("order-002", "user-42", 9999)
except RuntimeError as e:
    assert repo2.find_by_id("order-002") is None  # order not saved if payment fails
    print(f"✅ Payment failure handled cleanly — order not persisted: {e}")

---
## 11 · Event-Driven, CQRS & Event Sourcing

### 🧠 Mental Model — *The Newspaper vs The Phonebook*

> **A Phonebook (CRUD) is current state: look up who lives at an address right now. A Newspaper archive (Event Sourcing) is history: every event that ever happened, from which you can reconstruct any past state. CQRS says: use the phonebook for lookups (query), use the newspaper archive for writing the record (command).**

**WHY**:
- **Event-driven**: decouples producers from consumers — an order service doesn't need to know about email, inventory, analytics; it just emits `OrderPlaced`
- **CQRS**: reads and writes have different shapes; forcing them through one model makes both worse. A write needs an `Order` aggregate; a read needs a denormalized view with customer name and product images
- **Event Sourcing**: the only source of truth is the immutable event log — you get a free audit trail, time travel, and the ability to build new read projections from history

**WHAT**:
```
Event-Driven:   OrderService ---publishes--> [Kafka topic: orders] ---consumed-by--> EmailService
                                                                    ---consumed-by--> InventoryService
                                                                    ---consumed-by--> AnalyticsService

CQRS:           Write path: Command → Aggregate → Event Store
                Read path:  Event Store → Projector → Read Model (denormalized) → Query

Event Sourcing: State = replay(all events for this ID)
                Store: append only [{event_type, data, timestamp}, ...]
```

**WHEN**:
- Use event-driven when services need loose coupling and you have spiky/high-volume async work
- Use CQRS when read and write models diverge significantly (e.g., write = raw order data, read = denormalized dashboard)
- Use event sourcing when audit trails are required (finance, healthcare, legal) or you need temporal queries
- **Don't use** event sourcing for simple CRUD — the complexity is not worth it

### ⚠️ The Gotcha: At-Least-Once Delivery

> Kafka/SQS delivers messages **at least once**. Your consumer MUST be idempotent. The canonical fix: store a processed-event-id table and `UPSERT` based on it.

### 🌍 Where This Is Seen in Frameworks

| Pattern | Framework/Tool |
|---|---|
| Event bus | Kafka, RabbitMQ, AWS SQS/SNS, Redis Pub/Sub |
| Event sourcing | EventStoreDB, Marten (.NET), `eventsourcing` Python lib |
| CQRS in Python | `commands.py` + `queries.py` split; Axon Framework (Java) |
| Saga pattern | `choreography` (events) vs `orchestration` (saga orchestrator) |
| Celery tasks | Async task queue = lightweight event-driven |

In [ ]:
"""
Event Sourcing + CQRS — from scratch, runnable.

Demonstrates:
- Append-only event store
- State reconstruction by replaying events
- Separate read model (projection) vs write model (aggregate)
- Idempotent consumer pattern
"""
from __future__ import annotations
from dataclasses import dataclass
import uuid
import time


# ─── Events (immutable facts) ─────────────────────────────────────────────────
@dataclass(frozen=True)
class Event:
    event_id: str
    aggregate_id: str
    event_type: str
    data: dict
    timestamp: float = field(default_factory=time.time)


# ─── Event Store (append-only) ────────────────────────────────────────────────
class EventStore:
    def __init__(self):
        self._log: list[Event] = []  # THE source of truth — never mutated, only appended

    def append(self, event: Event) -> None:
        self._log.append(event)

    def get_events(self, aggregate_id: str) -> list[Event]:
        return [e for e in self._log if e.aggregate_id == aggregate_id]

    @property
    def all_events(self) -> list[Event]:
        return list(self._log)


# ─── Aggregate (write model — reconstructed by replaying events) ──────────────
class BankAccount:
    """
    State is never stored directly — only the events that produced it.
    To know the current balance, replay all events.
    """
    def __init__(self, account_id: str):
        self.account_id = account_id
        self.balance = 0
        self.owner = None
        self._version = 0

    @classmethod
    def from_events(cls, account_id: str, events: list[Event]) -> BankAccount:
        """Reconstruct state by replaying all events — this is event sourcing."""
        account = cls(account_id)
        for event in events:
            account._apply(event)
        return account

    def _apply(self, event: Event) -> None:
        if event.event_type == "AccountOpened":
            self.owner = event.data["owner"]
            self.balance = event.data["initial_deposit"]
        elif event.event_type == "MoneyDeposited":
            self.balance += event.data["amount"]
        elif event.event_type == "MoneyWithdrawn":
            self.balance -= event.data["amount"]
        self._version += 1

    # Command methods: validate, then emit events (no direct state mutation)
    def open(self, store: EventStore, owner: str, initial_deposit: int):
        event = Event(str(uuid.uuid4()), self.account_id, "AccountOpened",
                      {"owner": owner, "initial_deposit": initial_deposit})
        store.append(event)
        self._apply(event)

    def deposit(self, store: EventStore, amount: int):
        event = Event(str(uuid.uuid4()), self.account_id, "MoneyDeposited", {"amount": amount})
        store.append(event)
        self._apply(event)

    def withdraw(self, store: EventStore, amount: int):
        if amount > self.balance:
            raise ValueError(f"Insufficient funds: {self.balance} < {amount}")
        event = Event(str(uuid.uuid4()), self.account_id, "MoneyWithdrawn", {"amount": amount})
        store.append(event)
        self._apply(event)


# ─── Read Model / Projection (CQRS query side) ────────────────────────────────
class AccountSummaryProjection:
    """Denormalized read model — optimized for queries, rebuilt from events."""
    def __init__(self):
        self._summaries: dict[str, dict] = {}
        self._processed_events: set[str] = set()   # idempotency

    def process(self, event: Event) -> None:
        if event.event_id in self._processed_events:
            return  # idempotent — safe to replay
        self._processed_events.add(event.event_id)

        aid = event.aggregate_id
        if event.event_type == "AccountOpened":
            self._summaries[aid] = {"owner": event.data["owner"],
                                     "balance": event.data["initial_deposit"],
                                     "tx_count": 0}
        elif event.event_type == "MoneyDeposited":
            self._summaries[aid]["balance"] += event.data["amount"]
            self._summaries[aid]["tx_count"] += 1
        elif event.event_type == "MoneyWithdrawn":
            self._summaries[aid]["balance"] -= event.data["amount"]
            self._summaries[aid]["tx_count"] += 1

    def query(self, account_id: str) -> dict:
        return self._summaries.get(account_id, {})


# ─── Demo ─────────────────────────────────────────────────────────────────────
store = EventStore()
acc = BankAccount("acc-001")

acc.open(store, "Alice", 1000)
acc.deposit(store, 500)
acc.withdraw(store, 200)

print("=== Event Log (source of truth) ===")
for e in store.all_events:
    print(f"  [{e.event_type}] {e.data}")

print("\n=== Current state from aggregate ===")
print(f"  Balance: {acc.balance} | Version: {acc._version}")

# Reconstruct from scratch (proves events are the source of truth)
replayed = BankAccount.from_events("acc-001", store.get_events("acc-001"))
print(f"  Replayed balance: {replayed.balance} (matches: {replayed.balance == acc.balance})")

# Build CQRS read model
projection = AccountSummaryProjection()
for event in store.all_events:
    projection.process(event)
    projection.process(event)  # simulate duplicate delivery — idempotent!

summary = projection.query("acc-001")
print("\n=== CQRS Read Model ===")
print(f"  {summary}")
print(f"✅ Balance matches aggregate: {summary['balance'] == acc.balance}")

---
## 12 · Real-World Case Studies

### 🏦 Case Study 1: Stripe — Payments at Scale

**The Problem**: Process 250M+ transactions/year with zero double-charges, sub-200ms p99 latency, 99.999% availability.

**Key Architectural Decisions**:
1. **Idempotency keys** (mandatory for POST endpoints that charge): stored in Postgres with UNIQUE constraint
2. **CP consistency** for the ledger: Postgres, synchronous replication, strong consistency over availability
3. **Async for non-critical paths**: webhooks, email receipts delivered via queues
4. **Request IDs** on every response for correlation across retries
5. **Rate limiting per API key** with a sliding window algorithm (stored in Redis)

**Mental model**: Stripe treats every API request as a distributed transaction. The idempotency key IS the distributed transaction coordinator.

---

### 🎬 Case Study 2: Netflix — Streaming at Scale

**The Problem**: 200M+ subscribers, 15% of global internet bandwidth, zero buffering.

**Key Architectural Decisions**:
1. **Open Connect (CDN)**: Netflix ships hard drives to ISPs; 95% of traffic served from edge
2. **Chaos Engineering** (Chaos Monkey): deliberately kill production servers to test resilience
3. **Circuit breakers everywhere** (Hystrix): one slow microservice can't cascade
4. **AP for recommendations**: using Cassandra (AP) — slightly stale recommendations are fine
5. **CP for billing**: Postgres — staleness = double-charge = customer complaint
6. **Pre-encoded video variants**: each video encoded at 20+ bitrates in advance

**Mental model**: Netflix is an elaborate CDN problem disguised as a software problem.

---

### 🐦 Case Study 3: Twitter/X — Social Graph at Scale

**The Problem**: 300M DAU, celebrity tweets to 50M followers must appear in timelines in <5 seconds.

**Key Architectural Decisions**:
1. **Hybrid fan-out**: pre-compute timelines for most users (write-time fan-out); pull-on-read for celebrity followers (avoid writing 50M entries per tweet)
2. **Redis for timelines**: each user's timeline = a Redis sorted set (score = tweet timestamp)
3. **Cassandra for tweet storage**: AP, massive write throughput, time-series data fits its data model
4. **FlockDB for social graph**: custom graph DB for follower/following queries
5. **Finagle**: Twitter's own RPC framework (inspired gRPC's design)

**Back-of-envelope that drove the design**:
```
300M DAU × 200 average followers = 60B timeline inserts/day = 700K inserts/sec
Cannot be done synchronously → async fan-out workers
Katy Perry (146M followers) tweets → 146M Redis inserts → too slow!
Solution: don't pre-compute for celebrities; pull their tweets at read time and merge
```

---

### 🚗 Case Study 4: Uber — Real-Time Location at Scale

**The Problem**: Match drivers to riders in <10s in 10,000 cities. Drivers send GPS every 4s = ~2M writes/min.

**Key Architectural Decisions**:
1. **Geospatial index** (Google S2 cells / H3): divide map into hexagons, find nearby drivers efficiently
2. **Apache Kafka for location events**: 2M GPS events/min, multiple consumers (matching, ETA, surge pricing)
3. **CQRS for surge pricing**: read model (surge map) built from event stream, never blocking the write path
4. **PostgreSQL + PostGIS**: geospatial queries with spatial indexes for trip history
5. **Domain isolation**: pricing, matching, driver supply are independent services

**Mental model**: Uber is a geospatial stream processing problem. The core data structure is a map of hexagons with driver density counts, updated in real time from a Kafka stream.

---
## 13 · The Interview Playbook

### Step-by-Step Framework for Any System Design Interview

**Step 1 — Clarify (3-5 minutes)**
```
"Before I dive in, a few clarifying questions:"
→ Expected scale? (DAU, peak QPS)
→ Read/write ratio?
→ Latency requirements? (p99 target)
→ Consistency requirements? (is stale data acceptable?)
→ Global or single-region?
→ What's in scope? (just design, or also operation?)
```

**Step 2 — Estimate (2-3 minutes)**
```
State the math out loud — shows you can anchor design in reality:
→ "100M DAU × 10 requests/day = 1B requests/day = ~12K RPS average, ~60K peak"
→ "At 1KB per request = 60MB/s bandwidth — CDN likely needed"
→ "5 years × 365 days × 1B records × 1KB = 1.8PB — object store, not relational"
→ "This is write-heavy (10:1 write/read ratio) → optimize for write throughput, batch reads"
```

**Step 3 — Define the API Contract**
```python
# Write the key endpoints first — they constrain everything downstream
POST /tweets          body: {text, media_ids} → {tweet_id}
GET  /timeline        params: ?user_id&cursor → {tweets: [...], next_cursor}
GET  /tweet/{id}      → {tweet}
```

**Step 4 — Data Model**
```
→ What are the entities? (Tweet, User, Follow relationship)
→ SQL vs NoSQL? "Tweets are time-series with high write rate → Cassandra; user data is relational → Postgres"
→ Shard key? "Shard tweets by user_id — most queries are per-user"
```

**Step 5 — High-Level Design**
```
Client → CDN → Load Balancer → API servers (stateless) → Redis cache → DB
                                      ↓
                              Kafka (async fan-out)
                                      ↓
                         Timeline workers → Redis timeline cache
```

**Step 6 — Deep Dive (the bottleneck)**
```
→ "The bottleneck is timeline fan-out for celebrities. Here's how I'd solve it..."
→ Name the pattern: hybrid fan-out, consistent hashing for cache routing, etc.
```

**Step 7 — Operate It**
```
→ Monitoring: request rate, p99 latency, error rate (the golden signals)
→ Failure modes: what if Redis goes down? (fall back to DB with rate limiting)
→ Deployment: blue-green, canary
→ Data backup/recovery: RPO and RTO
```

### ⚠️ Common Interview Mistakes

| Mistake | Better approach |
|---|---|
| Jumping to microservices immediately | Start simple (modular monolith), justify split |
| Using Redis for everything | Justify each cache: what's hot, what's the TTL, what's the invalidation? |
| Ignoring failure modes | Always ask: what happens when X goes down? |
| Saying "we can use Kafka" without explaining why | Explain: what's the consumer, what's the durability need, why not a queue? |
| Sharding immediately | Start with replication; shard only when numbers demand it |
| Missing idempotency for mutation endpoints | Always ask: what happens if this is retried? |

In [ ]:
"""
Interview framework cheat sheet — run to print a structured interview guide.
Bring this mental checklist to every system design session.
"""

INTERVIEW_PLAYBOOK = """
╔══════════════════════════════════════════════════════════════════╗
║           SYSTEM DESIGN INTERVIEW PLAYBOOK                      ║
╠══════════════════════════════════════════════════════════════════╣
║ 1. CLARIFY  (5 min)                                             ║
║    □ DAU / peak QPS                                             ║
║    □ Read:write ratio                                           ║
║    □ p99 latency SLA                                            ║
║    □ Consistency: strong or eventual ok?                        ║
║    □ Global or single region?                                   ║
╠══════════════════════════════════════════════════════════════════╣
║ 2. ESTIMATE (3 min)                                             ║
║    □ avg RPS = requests/day ÷ 86,400                            ║
║    □ peak RPS = avg × 3-5×                                      ║
║    □ storage/year = records/day × 365 × record_size             ║
║    □ bandwidth = avg_rps × payload_size                         ║
║    → Identify: is this read-heavy? write-heavy? storage-bound?  ║
╠══════════════════════════════════════════════════════════════════╣
║ 3. API CONTRACT                                                 ║
║    □ Write the key 3-5 endpoints                                ║
║    □ Include: idempotency key for mutations? pagination?        ║
╠══════════════════════════════════════════════════════════════════╣
║ 4. DATA MODEL                                                   ║
║    □ Entities, relationships, access patterns                   ║
║    □ SQL vs NoSQL (justify by access pattern, consistency need) ║
║    □ Shard key (high cardinality, matches queries)              ║
╠══════════════════════════════════════════════════════════════════╣
║ 5. HIGH-LEVEL DESIGN                                            ║
║    Client→CDN→LB→App (stateless)→Cache→DB                      ║
║    + Async queue for slow/spiky work                            ║
╠══════════════════════════════════════════════════════════════════╣
║ 6. DEEP DIVE (bottleneck)                                       ║
║    □ Name the bottleneck from your estimate                     ║
║    □ Apply the right pattern: cache / shard / async / CB / etc  ║
╠══════════════════════════════════════════════════════════════════╣
║ 7. OPERATE IT                                                   ║
║    □ Monitoring: rate, latency, errors (golden signals)         ║
║    □ Failure modes: what if cache/DB/queue goes down?           ║
║    □ Deployment: canary, blue-green                             ║
║    □ RPO / RTO: backup and recovery                             ║
╠══════════════════════════════════════════════════════════════════╣
║ RED FLAGS to avoid:                                             ║
║  ✗ Jumping to microservices without justification              ║
║  ✗ No idempotency on mutation endpoints                         ║
║  ✗ Ignoring failure modes                                       ║
║  ✗ Using Kafka without explaining the consumer                  ║
║  ✗ Sharding before replication                                  ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(INTERVIEW_PLAYBOOK)